# Reel Pipeline (PowerShell + Speechma)
Run top-to-bottom. This notebook builds reel video, voice, captions, and branded final output.


## Setup
Load PowerShell cell magic (one-time per session).


## Step 1: Initialize Paths and Folders
Creates reel workspace folders and shared variables used by later cells.


In [1]:
# Initialize dynamic reel paths and ensure base folders exist
from pathlib import Path
from datetime import datetime
import subprocess

PROJECT_ROOT = Path(r"C:\Users\saura\Documents\youtubeVideoAgent")
TOPIC = "corey-wayne"
now = datetime.now()
DATE = now.strftime("%Y-%m-%d")
HOUR = now.strftime("%H")
REEL_ROOT = PROJECT_ROOT / "assets" / "reels" / f"{DATE}_{HOUR}_{TOPIC}"
FINAL_DIR = REEL_ROOT / "final"
CAPTIONS_DIR = REEL_ROOT / "captions"
VOICE_DIR = REEL_ROOT / "voice"
SCRIPT_DIR = REEL_ROOT / "script"
META_DIR = REEL_ROOT / "meta"
SOURCE = Path(r"C:\Users\saura\Downloads\grok-folder-1")
AUDIO = VOICE_DIR / "voice_v1.mp3"
SCRIPT_FROM_CELL = SCRIPT_DIR / "script_from_cell.txt"
INPUT_FROM_CELL = VOICE_DIR / "speechma_input_from_cell.txt"
SETTINGS_JSON = META_DIR / "speechma_settings.json"
FFMPEG = PROJECT_ROOT / "tools" / "ffmpeg" / "ffmpeg-8.1.1-essentials_build" / "bin" / "ffmpeg.exe"
FFPROBE = PROJECT_ROOT / "tools" / "ffmpeg" / "ffmpeg-8.1.1-essentials_build" / "bin" / "ffprobe.exe"
CONCAT = SOURCE / "concat.txt"
STITCHED_VIDEO = FINAL_DIR / "scenes_stitched.mp4"
VOICED_VIDEO = FINAL_DIR / "scenes_stitched_voiced.mp4"
WORDS = CAPTIONS_DIR / "word_timestamps.json"
SRT = CAPTIONS_DIR / "captions.srt"
ASS = CAPTIONS_DIR / "captions.ass"
FINAL_OUTPUT = FINAL_DIR / "final_captioned.mp4"
BRANDED_OUTPUT = FINAL_DIR / "final_captioned_branded.mp4"
LOGOS_DIR = PROJECT_ROOT / "assets" / "branding" / "logos"
SELECTED_LOGO_PATH = LOGOS_DIR / "logo_rounded_more.png"
for d in [REEL_ROOT, FINAL_DIR, CAPTIONS_DIR, VOICE_DIR, SCRIPT_DIR, META_DIR, LOGOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f'Reel root: {REEL_ROOT}')
print('Folders ready.')


print(f'Logos folder: {LOGOS_DIR}')
print(f'Default selected logo: {SELECTED_LOGO_PATH}')



Reel root: C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-16_14_corey-wayne
Folders ready.
Logos folder: C:\Users\saura\Documents\youtubeVideoAgent\assets\branding\logos
Default selected logo: C:\Users\saura\Documents\youtubeVideoAgent\assets\branding\logos\logo_rounded_more.png


## Step 2: Enter Speechma Narration Text
Paste text in the output textbox and click **Save Text**.


In [2]:
# UI: capture narration text and save default voice/effect settings
from scripts.reel_notebook_cells import step2_ui

required = ['PROJECT_ROOT','REEL_ROOT','VOICE_DIR','SCRIPT_DIR','META_DIR','AUDIO','SCRIPT_FROM_CELL','INPUT_FROM_CELL','SETTINGS_JSON']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

for d in [REEL_ROOT, VOICE_DIR, SCRIPT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Using reel root: {REEL_ROOT}')
step2_ui(REEL_ROOT, SCRIPT_FROM_CELL, INPUT_FROM_CELL, SETTINGS_JSON)



Using reel root: C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-16_14_corey-wayne


## Step 3: Generate Voice with Speechma API
Uses saved text from Step 2 and produces `voice/voice_v1.mp3` in the reel workspace.


In [3]:
# Generate voice_v1.mp3 via local Speechma API using saved settings
from scripts.reel_notebook_cells import run_step3

required = ['PROJECT_ROOT','REEL_ROOT','VOICE_DIR','SCRIPT_DIR','META_DIR','AUDIO','SCRIPT_FROM_CELL','INPUT_FROM_CELL','SETTINGS_JSON']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

for d in [REEL_ROOT, VOICE_DIR, SCRIPT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Using reel root: {REEL_ROOT}')
data = run_step3(PROJECT_ROOT, REEL_ROOT, VOICE_DIR, AUDIO, SCRIPT_FROM_CELL, SETTINGS_JSON)
data



Using reel root: C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-16_14_corey-wayne


{'ok': True,
 'pageId': 4,
 'scriptPath': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-16_14_corey-wayne\\script\\script_from_cell.txt',
 'inputPath': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-16_14_corey-wayne\\voice\\speechma_input_v1.txt',
 'outputVoicePath': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-16_14_corey-wayne\\voice\\voice_v1.mp3',
 'proofDir': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-16_14_corey-wayne\\analysis\\speechma_proof_api',
 'downloadedFile': 'C:\\Users\\saura\\Downloads\\speechma_audio_Brian Multilingual_at_2_28_01 PM_on_May_16th_2026.mp3',
 'durationSeconds': 66.6,
 'settings': {'pitch': 0, 'speed': 15, 'volume': 200},
 'matchPhraseUsed': 'Most people ruin relationships by trying to fast-forward them.',
 'topic': None,
 'sourceRoot': None,
 'evidencePath': None}

## Step 3.5: Calculate Scene Count from Voice Length
Uses `voice/voice_v1.mp3` duration and computes rounded scene counts for 6s and 10s pacing.


In [4]:
required = ['FFPROBE', 'AUDIO']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Compute recommended scene counts from voice_v1.mp3 duration
raw = subprocess.check_output([str(FFPROBE), '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(AUDIO)], text=True).strip()
duration_sec = float(raw)
scenes_6s = round(duration_sec / 6)
scenes_10s = round(duration_sec / 10)
print(f'Voice duration: {duration_sec:.2f} sec')
print(f'6s scenes needed: {scenes_6s}')
print(f'10s scenes needed: {scenes_10s}')
{'voice_duration_sec': duration_sec, 'scene_count_6s': scenes_6s, 'scene_count_10s': scenes_10s}



Voice duration: 66.60 sec
6s scenes needed: 11
10s scenes needed: 7


{'voice_duration_sec': 66.6, 'scene_count_6s': 11, 'scene_count_10s': 7}

## Step 4: Build Base Video and Mux Voice
Creates concat file, stitches scenes, then combines stitched video with generated voice.


In [9]:
required = ['PROJECT_ROOT', 'REEL_ROOT', 'SOURCE']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 4a: Build concat file (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step4_video.py'),
    '--action','concat',
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
    '--source', str(SOURCE),
], check=True)



CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step4_video.py', '--action', 'concat', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne', '--source', 'C:\\Users\\saura\\Downloads\\grok-folder-1'], returncode=0)

In [10]:
required = ['PROJECT_ROOT', 'REEL_ROOT', 'SOURCE']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 4b: Stitch scenes (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step4_video.py'),
    '--action','stitch',
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
    '--source', str(SOURCE),
], check=True)



CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step4_video.py', '--action', 'stitch', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne', '--source', 'C:\\Users\\saura\\Downloads\\grok-folder-1'], returncode=0)

In [11]:
required = ['PROJECT_ROOT', 'REEL_ROOT', 'SOURCE']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 4c: Mux stitched video with voice (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step4_video.py'),
    '--action','mux',
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
    '--source', str(SOURCE),
], check=True)



CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step4_video.py', '--action', 'mux', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne', '--source', 'C:\\Users\\saura\\Downloads\\grok-folder-1'], returncode=0)

## Step 5: Generate Captions
Creates word timestamps, SRT, and animated ASS captions.


In [12]:
required = ['PROJECT_ROOT', 'REEL_ROOT']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 5a: Generate captions (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step5_captions.py'),
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
    '--preset', 'logicloom_ref',
], check=True)

CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step5_captions.py', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne', '--preset', 'logicloom_ref'], returncode=0)

In [13]:
required = ['WORDS', 'SRT', 'ASS']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 5b: Quick check generated caption files
print('WORDS:', WORDS.exists(), WORDS)
print('SRT:  ', SRT.exists(), SRT)
print('ASS:  ', ASS.exists(), ASS)



WORDS: True C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-15_19_corey-wayne\captions\word_timestamps.json
SRT:   True C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-15_19_corey-wayne\captions\captions.srt
ASS:   True C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-15_19_corey-wayne\captions\captions.ass


In [14]:
required = ['SRT']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 5c: Preview first lines of SRT
if SRT.exists():
    print('\n'.join(SRT.read_text(encoding='utf-8', errors='ignore').splitlines()[:20]))
else:
    print('SRT not found yet.')



1
00:00:00,000 --> 00:00:01,020
Stop. If you

2
00:00:01,020 --> 00:00:01,560
skip this one

3
00:00:01,560 --> 00:00:02,540
food, your bones

4
00:00:02,540 --> 00:00:03,240
could start failing

5
00:00:03,240 --> 00:00:04,040
you before 40.



## Step 5.5: Select Branding Logo
Pick logo from central folder for Step 6 watermark.


In [15]:
# Select logo from central logos folder for Step 6 watermark
import ipywidgets as widgets
from IPython.display import display

required = ['PROJECT_ROOT','LOGOS_DIR']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

LOGOS_DIR.mkdir(parents=True, exist_ok=True)
logo_files = sorted([p for p in LOGOS_DIR.iterdir() if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}])
if not logo_files:
    raise RuntimeError(f"No logo files found in {LOGOS_DIR}. Add files first.")

default_logo = SELECTED_LOGO_PATH if 'SELECTED_LOGO_PATH' in globals() and Path(SELECTED_LOGO_PATH).exists() else logo_files[0]
dd = widgets.Dropdown(options=[(p.name, str(p)) for p in logo_files], value=str(default_logo), description='Logo:', layout=widgets.Layout(width='80%'))
btn = widgets.Button(description='Use This Logo', button_style='success', icon='check')
out = widgets.Output()

def on_pick(_):
    global SELECTED_LOGO_PATH
    SELECTED_LOGO_PATH = Path(dd.value)
    with out:
        out.clear_output()
        print(f'Selected logo: {SELECTED_LOGO_PATH}')

btn.on_click(on_pick)
display(widgets.VBox([widgets.HTML(f"<b>Logos folder:</b> <code>{LOGOS_DIR}</code>"), dd, btn, out]))

# initialize selected logo from current dropdown value
SELECTED_LOGO_PATH = Path(dd.value)
print(f'Current selected logo: {SELECTED_LOGO_PATH}')


Current selected logo: C:\Users\saura\Documents\youtubeVideoAgent\assets\branding\logos\relationship_playbook.png


## Step 6: Burn Captions and Add Branding
Burns ASS captions onto video, adds watermark, and prints final output path.


In [16]:
required = ['PROJECT_ROOT', 'REEL_ROOT']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 6a: Burn ASS captions (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step6_finalize.py'),
    '--action','burn',
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
], check=True)



CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step6_finalize.py', '--action', 'burn', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne'], returncode=0)

In [17]:
required = ['PROJECT_ROOT', 'REEL_ROOT', 'SELECTED_LOGO_PATH']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 + Step 5.5 required first. Missing variables: {missing}")

# Step 6b: Apply branding watermark (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step6_finalize.py'),
    '--action','watermark',
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
    '--logo-path', str(SELECTED_LOGO_PATH),
], check=True)




CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step6_finalize.py', '--action', 'watermark', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne', '--logo-path', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\branding\\logos\\relationship_playbook.png'], returncode=0)

In [18]:
required = ['PROJECT_ROOT', 'REEL_ROOT']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 6c: Print final output path (delegated to script)
subprocess.run([
    'python', str(PROJECT_ROOT / 'scripts' / 'reel_step6_finalize.py'),
    '--action','print',
    '--project-root', str(PROJECT_ROOT),
    '--reel-root', str(REEL_ROOT),
], check=True)



CompletedProcess(args=['python', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\scripts\\reel_step6_finalize.py', '--action', 'print', '--project-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent', '--reel-root', 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-15_19_corey-wayne'], returncode=0)

In [19]:
required = ['FINAL_OUTPUT', 'BRANDED_OUTPUT']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"Step 1 required first. Missing variables: {missing}")

# Step 6d: Sanity check final files
print('Final captioned:', FINAL_OUTPUT.exists(), FINAL_OUTPUT)
print('Final branded: ', BRANDED_OUTPUT.exists(), BRANDED_OUTPUT)



Final captioned: True C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-15_19_corey-wayne\final\final_captioned.mp4
Final branded:  True C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-15_19_corey-wayne\final\final_captioned_branded.mp4
